# Lista 5 — Zadanie 2: Klasyfikacja encoder-only (HerBERT) (20 pkt)

In [1]:
import sys

!{sys.executable} -m pip install -q torch transformers datasets scikit-learn pandas

In [2]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import torch
from transformers import pipeline
from tqdm.auto import tqdm

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions, print_evaluation

## Krok 1: Ładowanie danych

In [3]:
examples = load_polemo_test()
sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]

print(f"Liczba próbek do klasyfikacji: {len(sentences)}")

Liczba próbek do klasyfikacji: 614


## Krok 2: Ładowanie modelu encoder-only

In [ ]:
ENCODER_MODEL = "Voicelab/herbert-base-cased-sentiment"
device = 0 if torch.cuda.is_available() else -1

print(f"Urządzenie: {'GPU' if device == 0 else 'CPU'}")

sentiment_pipeline = pipeline(
    "text-classification",
    model=ENCODER_MODEL,
    device=device,
)

## Krok 3: Klasyfikacja i mapowanie etykiet

In [ ]:
def predict_encoder(sentences, pipe):
    predictions = []
    unmapped = []

    raw_outputs = pipe(sentences, batch_size=16, truncation=True, max_length=512)

    for text, output in zip(sentences, raw_outputs):
        label_text = output["label"]
        mapped = map_text_to_class(label_text)

        if mapped is None:
            unmapped.append((text[:80], label_text))
            mapped = "neutral"

        predictions.append(mapped)

    if unmapped:
        print(f"Uwaga: {len(unmapped)} etykiet nie udało się zmapować (użyto fallback 'neutral')")
        for text, label in unmapped[:3]:
            print(f"  '{label}' dla: {text}...")

    return predictions


y_pred = predict_encoder(sentences, sentiment_pipeline)

## Krok 4: Ewaluacja wyników

In [ ]:
results = evaluate_predictions(y_true, y_pred)
print_evaluation(results, title=f"HerBERT baseline — {ENCODER_MODEL}")

## Krok 5: Przykłady błędnych klasyfikacji

In [ ]:
print("Przykłady pomyłek modelu (max 5):")
shown = 0
for ex, pred in zip(examples, y_pred):
    if ex["class"] != pred and shown < 5:
        print(f"\nPrawda: {ex['class']} | Predykcja: {pred}")
        print(ex["sentence"][:250] + "...")
        shown += 1

: 